In [12]:
#alle relevanten imports:

import pandas as pd
import os
import glob
from Bio import SeqIO
from pathlib import Path
from scipy.stats import chi2_contingency 




In [3]:

seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset = seq_regions)
    .dropna(subset = cf_regions)
    .drop_duplicates(subset = seq_regions) 
)

antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt



In [5]:
#FASTA erzeugung für CDRs in Subgruppen
#mit PDB und Antigennamen


# Zielordner für die FASTAs
fasta_dir = "data\\MMseqs2\\MMseqs2_Fasta"
os.makedirs(fasta_dir, exist_ok=True)       #erstellt Verzeichnis, wenn bereits exist, passiert nichts



# Für jede Region: Länge berechnen, nach Länge gruppieren und FASTA-Dateien schreiben
for region in seq_regions:
    length_col = f"{region}_length"         #Aus Namen einer sequenzregion ein bennater String für neue Spalte
    df[length_col] = df[region].str.len()   #Länge der sequenzstring berechnet und Werte in Spalte genommen

    for length, subdf in df.groupby(length_col):            #splits dataframe into sub dataframens based on length
        fasta_path = os.path.join(fasta_dir, f"{region}_{length}.fasta")    
        with open(fasta_path, 'w') as out:                  #opens data for creation, if pre existing rewriten
            for i, row in enumerate(subdf.itertuples(index=False), start=1):    #each line as tuple and numerated 
                seq     = getattr(row, region)
                pdb     = getattr(row, 'pdb')              
                antigen = getattr(row, 'antigen_name').replace(' ', '_') #antigenname is taken and space replaced with lowered line-->no space in Fasta header
                Hchain = getattr(row, 'Hchain')
                Lchain = getattr(row, 'Lchain')

                header = f"seq_{i} {pdb}|{antigen}|{Hchain}|{Lchain}"
                
                out.write(f">{header}\n{seq}\n")




In [ ]:
#Clustern in WSL mit MMSeqs2
#Reine Dokumentation




fasta_dir="data/MMseqs2/MMseqs2_Fasta"
outdir="data/MMseqs2/MMseqs2_cluster"
mkdir -p "$outdir"                      #-p: if the folder is pre existing no error

for fasta in "${fasta_dir}"/SEQ_*_*.fasta; do     #loop for all FASTA files with specific name
  
  
  #splits dataname, 
  base=$(basename "$fasta" .fasta)     #deletes path and ending
  region=${base%_*}                    #cuts after last subline, just region
  length=${base##*_}                   #gets everything after last subline, length

  #Festlegung Datenbank namen und ausgabe der tsv
  db="${region}_${length}_db"
  clu="${region}_${length}_clu"
  tsv="${outdir}/${region}_${length}_clusters.tsv"

  # FASTA → MMseqs2-Datenbank  
  if ! mmseqs createdb "$fasta" "$db"; then
    continue
  fi

  #Clustering, ! if a MMSeqs2 is not able to cluster data (length=5) delete temporary data an move on 
  if ! mmseqs cluster \
        --min-seq-id 0.6 \
        -c 1 \
        --spaced-kmer-mode 0 \
        "$db" "$clu" tmp; then
    continue
  fi

  #TSV-Export  
  if ! mmseqs createtsv "$db" "$db" "$clu" "$tsv"; then         
    continue
  fi

  #clean up
  rm -rf tmp "$db" "$clu"       #deletes temporary tmp folder, db and clus
done


In [7]:
#adjusting order of the clustered data

cluster_folder = Path("data/MMseqs2/MMseqs2_cluster")

# loop for cluster folder
for path in cluster_folder.glob("*_clusters.tsv"):          #returns iterator over specific files
    

    # opens read mode and reads only first line/header
    with open(path, "r") as f:                              
        header_line = f.readline().rstrip("\n")     

    # loads rest into a df
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["seq_rep", "seq_member", "antigen_name"],        #column names are assigned
        dtype=str,                                              #all coluns as strings
        skiprows=1                                              #first line (header is skipped)
    )

    #extracting number indices
    df["rep_num"] = (
        df["seq_rep"]
          .str.extract(r"(\d+)$", expand=False)     #captures one or more digits at the end of the string, returs numeric number as a string
          .astype(int)                              #converts string into integer
    )
    df["member_num"] = (                            #same for member
        df["seq_member"]
          .str.extract(r"(\d+)$", expand=False)
          .astype(int)
    )

    #sorting df first rep_num then member_num based on integer
    df_sorted = df.sort_values(
        by=["rep_num", "member_num"],
        ascending=[True, True],                 #both in ascending order
        ignore_index=True,                      #discards old row indices
        #na_position="last"
    )

    #rewrite with "w" 
    with open(path, "w", newline="") as f:
        f.write(header_line + "\n")                                 #old header
        df_sorted[["seq_rep", "seq_member", "antigen_name"]] \
            .to_csv(f, sep="\t", header=False, index=False)         #sorts based on new order



In [ ]:
#adding pdb id to clustered data

cluster_dir = "data\\MMseqs2\\MMseqs2_cluster"
fasta_dir   = "data\\MMseqs2\\MMseqs2_Fasta"


cluster_files = glob.glob(os.path.join(cluster_dir, "SEQ_*_clusters.tsv"))      #finds all files with correct names within cluster dir
#os.path join: joins all path compononents into one


for cluster_file in cluster_files:
    base       = os.path.basename(cluster_file).replace("_clusters.tsv", ".fasta")      #os pathname choose correct data name and exchanges it to fast
    fasta_file = os.path.join(fasta_dir, base)                                          ##ads fasta to directory

    id2pdb     = {}                             #empty dictionary for ad2pdb and id2antigen
    id2antigen = {}
    for rec in SeqIO.parse(fasta_file, "fasta"):            #each sequence as seq
        parts = rec.description.split()                     #splits header at space tab in two parts
        if len(parts) > 1 and "|" in parts[1]:              #check wether there is a second part including |
            pdb, antigen = parts[1].split("|", 1)           #if so splits it again at |
        else:
            pdb, antigen = "", ""                           #else both strings are empty
        id2pdb[rec.id]     = pdb                            #safe splited pdb code 
        id2antigen[rec.id] = antigen                        #safe splited antigen

    #proof for pre-existing header
    with open(cluster_file, 'r') as f:
        first = f.readline().strip().split('\t')
    has_header = first[0] == "seq_rep"                      #reads first line in tsv data, first entry should be seq rep--True

    #loads cluster tsv in data frame
    df = pd.read_csv(
        cluster_file,
        sep="\t",
        header=None,
        names=["seq_rep", "seq_member"],                    
        usecols=[0, 1],                                     #reads only first two columns of the data (seq_rep/seq_member)
        skiprows=1 if has_header else 0,                    #skips first line if there is an header
        dtype=str,
        engine="python"                                     #use python parser
    )

    #add new columns
    df["antigen_name"] = df["seq_member"].map(id2antigen).fillna("")
    df["pdb_member"]     = df["seq_member"].map(id2pdb).fillna("")

    #rewrite data with new columns and header (including old seq_rep and seq member)
    df.to_csv(cluster_file, sep="\t", index=False)


In [ ]:
#adding pdb id, antigen_name, h chain, l chain


cluster_dir = "data\\MMseqs2\\MMseqs2_cluster"
fasta_dir   = "data\\MMseqs2\\MMseqs2_Fasta"

cluster_files = glob.glob(os.path.join(cluster_dir, "SEQ_*_clusters.tsv"))       #finds all files with correct names within cluster dir
#os.path join: joins all path compononents into one

for cluster_file in cluster_files:
    # korrespondierende Fasta-Datei ermitteln
    base       = os.path.basename(cluster_file).replace("_clusters.tsv", ".fasta")      #os pathname choose correct data name and exchanges it to fast
    fasta_file = os.path.join(fasta_dir, base)                                          #ads fasta to directory

    # empty Dictionaries für pdb, antigen, H- und L-Chain
    id2pdb     = {}
    id2antigen = {}
    id2Hchain  = {}
    id2Lchain  = {}

    # Header parsen
    for rec in SeqIO.parse(fasta_file, "fasta"):                                        #each sequence as seq
        
        parts = rec.description.split(maxsplit=1)
        if len(parts) > 1 and "|" in parts[1]:
            info    = parts[1].split("|")
            pdb     = info[0] if len(info) > 0 else ""
            antigen = info[1] if len(info) > 1 else ""
            Hchain  = info[2] if len(info) > 2 else ""
            Lchain  = info[3] if len(info) > 3 else ""
        else:
            pdb = antigen = Hchain = Lchain = ""

        id2pdb[rec.id]     = pdb
        id2antigen[rec.id] = antigen
        id2Hchain[rec.id]  = Hchain
        id2Lchain[rec.id]  = Lchain

    #add new columns
    with open(cluster_file, 'r') as f:                                                  #reads first line in tsv data, first entry should be seq rep--True
        first = f.readline().strip().split('\t')
    has_header = (first[0] == "seq_rep")

    # TSV einlesen
    df = pd.read_csv(
        cluster_file,
        sep="\t",
        header=None,
        names=["seq_rep", "seq_member"],
        usecols=[0, 1],                                                                 #reads only first two columns of the data (seq_rep/seq_member)
        skiprows=1 if has_header else 0,                                                #skips first line if there is an header
        dtype=str,
        engine="python"                                                                 #use python parser
    )

    # Neue Spalten befüllen
    df["antigen_name"] = df["seq_member"].map(id2antigen).fillna("")
    df["pdb_member"]   = df["seq_member"].map(id2pdb).fillna("")
    df["Hchain"]       = df["seq_member"].map(id2Hchain).fillna("")
    df["Lchain"]       = df["seq_member"].map(id2Lchain).fillna("")

    #rewrite data with new columns and header (including old seq_rep and seq member)
    df.to_csv(cluster_file, sep="\t", index=False)

In [ ]:
#adjusting order nach dem pdb id/ antigenname/Hchain/Lchain hinzugefügt wurden


cluster_folder = Path("data/MMseqs2/MMseqs2_cluster")

for path in cluster_folder.glob("*_clusters.tsv"):
    # 1) Alten Header (Spaltennamen) einlesen, falls du ihn behalten willst
    with open(path, "r") as f:
        header_line = f.readline().rstrip("\n")
    
    # 2) Den Rest als DataFrame einlesen – jetzt mit allen 6 Spalten
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["seq_rep", "seq_member", "antigen_name", "pdb_member", "Hchain", "Lchain"],
        skiprows=1,
        dtype=str,
        engine="python"
    )

    # 3) Hilfsspalten zum Sortieren extrahieren
    df["rep_num"] = (
        df["seq_rep"]
          .str.extract(r"(\d+)$", expand=False)
          .astype(int)
    )
    df["member_num"] = (
        df["seq_member"]
          .str.extract(r"(\d+)$", expand=False)
          .astype(int)
    )

    # 4) Sortieren nach rep_num und member_num
    df_sorted = df.sort_values(
        by=["rep_num", "member_num"],
        ascending=[True, True],
        ignore_index=True
    )

    # 5) Gewünschte Spaltenreihenfolge festlegen
    cols_order = ["seq_rep", "seq_member", "pdb_member", "antigen_name", "Hchain", "Lchain"]

    # 6) Neu schreiben mit Header und sortierten Daten
    with open(path, "w", newline="") as f:
        # Entweder den alten Header-Line nutzen:
        # f.write(header_line + "\n")
        # Oder einen neuen Header bauen:
        f.write("\t".join(cols_order) + "\n")

        # Daten ohne Index und ohne Zusatz-Header reinschreiben
        df_sorted[cols_order].to_csv(f, sep="\t", header=False, index=False)



In [9]:
#Enno Plakat:

#resultate zeigen, was hat geklappt bzw. was hat nicht so gut geklappt, was hat gar nicht funktioniert, warum hat es nicht funktioniert
#was sind dlie Mimitationen dieses clustering verfahren, warum nicht unbedingt so passend zu unserem Projekt
#sequenz identity und das aligment kurz ansprechen

#für Präsentation, warum MMSeqs2 ausprobiert

In [18]:
#Datei zum Vergleich von Vmeasure erstellen

BASE_DIR = "data\\MMseqs2\\MMseqs2_cluster"

cdr_regions = ["H1", "H2", "L1", "L2", "L3"]

def load_and_label(region):                         #definition of function
    parts = []                                      #creats storage list
    
    pattern = os.path.join(BASE_DIR, f"SEQ_{region}_*_clusters.tsv")    #builds file search pattern in base_dir with correct name
    for fn in glob.glob(pattern):                                       #loop for matching files
        length = os.path.basename(fn).split("_")[2]                     #extracts length of the filename, splited at each under score element (2), starting by zero
        prefix = f"{region}-{length}"                                   #each region is linked to length
        colname = f"CF_{region}"                                        #new column name
        
        df = pd.read_csv(fn, sep="\t", usecols=["seq_rep", "antigen_name", "pdb_member"])       #loads only necesarry columns (fn= tsv file into dataframe)
        df[colname] = df["seq_rep"].apply(lambda r: f"{prefix}-{r}")                            #creates new column, for each value in seq_rep member-->in which cluster is member located
        parts.append(df[["pdb_member", "antigen_name", colname]])       #only important columns are keept for merging
    
    if not parts:                                                       #in case no files are found
        return None
    
    out = pd.concat(parts, ignore_index=True).drop_duplicates(          #pd.contact() stacks all small dataframes into one, drop duplicates (same pdb entries with same antigen in different tsv (multiople cdrs) for each pdb and antigen combinantion only one line) if pdb and antigencombinantion is not identical it will be added
        subset=["pdb_member", "antigen_name"]
    )
    return out.set_index(["pdb_member", "antigen_name"])                #converts two columns into multiIndex on returned datafram


region_dfs = {}                                                         #regions df will map each region (H1,..) to corresponding df by load and label
for region in cdr_regions:
    df = load_and_label(region)                                         #returns df indexed by pdb_member and antigen name
    if df is not None:
        region_dfs[region] = df                                         #if load and labes found files-->its added into df name region



merged = None                                                           #merged holds joined dataframe, starts at none
for df in region_dfs.values():                                          #iterates over each regions dataframe
    merged = df.copy() if merged is None else merged.join(df, how="outer")  #only else part: if it is not none-->perform outer join of existinge merge with new df (keep all pdb_member/antigenname) or fill NAA


merged = (
    merged
    .reset_index()                                                      #turns index back into normal columns named pdb member and antigen_name
    .rename(columns={                                                   #renamed
        "pdb_member": "PDB_ID",
        "antigen_name": "antigen_name"
    })
)

cols = ["PDB_ID", "antigen_name"] + [f"CF_{r}" for r in cdr_regions]    #builds columnlist + appends CF_regions
cols = [c for c in cols if c in merged.columns]                         #filters out CF colums as H2-5 is not existing
merged = merged[cols]                                                   #reorder dataframe in order given by columns

#safe

merged.to_csv("data/MMseqs2/MMseqs2_summary_cluster.tsv", sep="\t", index=False) #exports file without panda indes


#Da clustering für SEQ H2-5, Seq L3-5 nicht erstellt wurden ist kommt es zu missing values im MMseqs2_summary cluster dokument

